#  This Marine Cold Spells (MCS) pipeline detect and calculte the MCS using *10th percentile threshold* which is standered MHW detection *Hobdey et at. (2016)* method.

In [ ]:
# primary requirements are the libraries and packages must be installed (python3, numpy, pandas, xarray, dask, bottleneck, netCDF4, cftime, matplotlib, marineHeatWaves)
# also fixes np.NaN to np.nan in marineHeatWaves module

import sys, platform
import numpy as np
import pandas as pd
import xarray as xr, dask, bottleneck, netCDF4, cftime
from pathlib import Path
from typing import List, Optional, Tuple, Dict, Union
import matplotlib.pyplot as plt
import os
import matplotlib.dates as mdates
import marineHeatWaves as mhw, re, pathlib
from importlib.metadata import version, PackageNotFoundError

# convert np.NAN to np.nan to avoid dependency issues
if not hasattr(np, "NaN"):
    np.NaN = np.nan  

# Check the package is working or not by printing the versions of the dependencies
try:
    mhw_ver = version('marineHeatWaves')
except PackageNotFoundError:
    mhw_ver = getattr(mhw, '__version__', 'unknown')
print('OK:', pd.__version__, xr.__version__, dask.__version__, 'mhw:', mhw_ver)
print('mhw module file:', getattr(mhw, '__file__', 'unknown'))

# check the marineHeatWaves module file and replace np.NaN with np.nan
p = pathlib.Path(mhw.__file__)
p.write_text(re.sub(r'\bnp\.NaN\b', 'np.nan', p.read_text()))

print(sys.executable)
print(platform.python_version())

In [ ]:
#check if coldspells is working in MHW package
t = pd.date_range("2000-01-01", periods=40, freq="D")
ords = np.array([d.toordinal() for d in t], dtype=int)
temp = 25 + np.sin(np.linspace(0, 6.28, 40)); temp[15:25] += 2

res, clim = mhw.detect(ords, temp, climatologyPeriod=[2000, 2000],
                       pctile=10, minDuration=5, joinAcrossGaps=True, coldSpells=True)
print("events:", len(res.get("time_start", [])))

# Verifies that core libs are importable and compatible with marineHeatWaves.
from importlib import import_module
import numpy as np
import marineHeatWaves as mhw

want = ["numpy", "pandas", "xarray", "dask", "bottleneck","netCDF4", "cftime", "marineHeatWaves"]
vers = {}
problems = []

for m in want:
    try:
        import_module(m)
        try:
            vers[m] = version(m)
        except PackageNotFoundError:
            mod = import_module(m)
            vers[m] = getattr(mod, "__version__", "unknown")
    except Exception as e:
        problems.append((m, str(e)))
        vers[m] = "IMPORT FAILED"

print("Package versions detailes:")
for k in want:
    print(f"{k:16s} : {vers[k]}")
if problems:
    print("\n[ENV WARNING] Some packages failed to import:")
    for m, msg in problems:
        print(f" - {m}: {msg}")

# marineHeatWaves.detect accepts ordinal ints + 1D temp array
try:
    # make a synthetic 40-day series with a warm spell
    days = np.array([pd.Timestamp("2000-01-01") + pd.Timedelta(i, "D") for i in range(40)])
    ords = np.array([d.toordinal() for d in days], dtype=int)
    temp = 25 + np.sin(np.linspace(0, 6.28, 40))
    temp[15:25] += 2.0  # warm event
    res, clim = mhw.detect(ords, temp, climatologyPeriod=[1991, 2020], pctile=90, minDuration=5, joinAcrossGaps=True, coldSpells= True)
    print("\nmarineHeatWaves.detect is working correctly.")
    print(f"Detected events: {len(res.get('time_start', []))}")
except Exception as e:
    print("\n[ENV ERROR] marineHeatWaves.detect detection failed:", e)

**Common code for Climatologoy, Threshold, Mean STT , SST Trend and MHW days correlation analysis**

In [ ]:
# Common code for Climatologoy, Threshold plots and analysis
#Run this code first before running other analysis codes

from glob import glob
from pathlib import Path
from typing import Dict, Tuple, Optional
import numpy as np, pandas as pd, xarray as xr
import matplotlib.pyplot as plt
import datetime as dt
import matplotlib.dates as mdates
import marineHeatWaves as mhw  

# convert np.NAN to np.nan to avoid dependency issues
import numpy as _np
if not hasattr(_np, "NaN"):
    _np.NaN = _np.nan

# File path
FILES_GLOB = "/home/Desktop/Noah_data_1982-2024_SST_daily_mean/sst.day.mean.*.nc"
VAR = "sst"
CLIM_YEARS: Tuple[int,int] = (1982, 2024)   
REGIONS: Dict[str, Dict[str, float]] = {   # modify the lat-lon bounds as per your region
    "Arabian Sea":    {"lon_min": 40.0, "lon_max": 78.0,  "lat_min": 0.0, "lat_max": 30.0},
    "Bay Of Bengal":   {"lon_min": 78.0, "lon_max": 110.0, "lat_min": 0.0, "lat_max": 30.0},
    "North Indian Ocean": {"lon_min": 40.0, "lon_max": 110.0, "lat_min": 0.0, "lat_max": 30.0},
}

SEASONS = {"DJF": [12, 1, 2],"MAM": [3, 4, 5],"JJA": [6, 7, 8],"SON": [9, 10, 11],}
season_order = ["DJF", "MAM", "JJA", "SON"] 
OUTDIR = Path("outputs"); OUTDIR.mkdir(parents=True, exist_ok=True)

# leap-aware reference year 
_ref = pd.date_range("2000-01-01", "2000-12-31", freq="D") 
_doy_to_month = pd.Series(_ref.month.values, index=np.arange(1, 367))
_month_to_doy0 = {m: (_doy_to_month[_doy_to_month == m].index.values - 1) for m in range(1, 13)}


def open_mfdataset(paths_glob: str, chunks={"time": 120}, engine: str = "netcdf4") -> xr.Dataset:
    paths = sorted(glob(paths_glob))
    if not paths:
        raise FileNotFoundError(f"No files match: {paths_glob}")
    print(f"[open] {len(paths)} files")
    return xr.open_mfdataset(paths, combine="by_coords", parallel=True, chunks=chunks, engine=engine)

def subset_box(ds: xr.Dataset, box: Dict[str, float]) -> xr.Dataset:
    latn = "lat" if "lat" in ds.coords else "latitude"
    lonn = "lon" if "lon" in ds.coords else "longitude"
    return ds.sel({latn: slice(box["lat_min"], box["lat_max"]),lonn: slice(box["lon_min"], box["lon_max"])})

def area_weighted_boxmean(da: xr.DataArray) -> xr.DataArray:
    latn = "lat" if "lat" in da.coords else "latitude"
    w = np.cos(np.deg2rad(da[latn]))
    return da.weighted(w).mean(dim=[latn, "lon" if "lon" in da.coords else "longitude"])

In [ ]:
# Long-term Climatology, 90th and 10th percentile Threshold of Arabian Sea, Bay Of Bengal, North Indian Ocean

def mhw_seas_thresh_doy(da_box: xr.DataArray, clim_years: Tuple[int,int]) -> Tuple[np.ndarray, np.ndarray]:

    t_index = da_box["time"].to_index()

    # safety-clip baseline to data span
    y0, y1 = max(clim_years[0], t_index.year.min()), min(clim_years[1], t_index.year.max())

    ords = np.array([d.toordinal() for d in t_index], dtype=int)
    temp = da_box.values.astype(float)

    # clim['seas'] and clim['thresh'] for the 90th percentile 
    res_90th, clim_90th = mhw.detect(ords, temp, climatologyPeriod=[int(y0), int(y1)], pctile=90, minDuration=5, joinAcrossGaps=True)
    seas_full   = np.asarray(clim_90th["seas"])
    thresh_90th_full = np.asarray(clim_90th["thresh"])

    # clim['seas'] and clim['thresh'] for the 10th percentile 
    res_10th, clim_10th = mhw.detect(ords, temp, climatologyPeriod=[int(y0), int(y1)], pctile=10, minDuration=5, joinAcrossGaps=True)
    thresh_10th_full = np.asarray(clim_10th["thresh"])

    # Group by day-of-year to get 366-length curves (leap-aware).
    doy = t_index.dayofyear.values
    seas_366   = np.full(366, np.nan, float)
    thresh_90th_366 = np.full(366, np.nan, float)
    thresh_10th_366 = np.full(366, np.nan, float)
    for d in range(1, 367): 
        m = (doy == d)
        if m.any():
            seas_366[d-1]   = np.nanmean(seas_full[m])
            thresh_90th_366[d-1] = np.nanmean(thresh_90th_full[m])
            thresh_10th_366[d-1] = np.nanmean(thresh_10th_full[m])
    return seas_366, thresh_90th_366, thresh_10th_366

# Load dataset
ds_annual= open_mfdataset(FILES_GLOB)
# Compute curves per region and store
curves_annual = {}  
for name, box in REGIONS.items():
    da = subset_box(ds_annual[[VAR]], box)[VAR]
    da_box = area_weighted_boxmean(da)
    seas_doy, thresh_90th_doy, thresh_10th_doy = mhw_seas_thresh_doy(da_box, CLIM_YEARS)
    curves_annual[name] = {"seas": seas_doy, "90th thresh": thresh_90th_doy, "10th thresh": thresh_10th_doy}

# Plotting three stacked panels with shared X-axis
x_dates = pd.date_range("2000-01-01", "2000-12-31", freq="D")  
fig, axes = plt.subplots(nrows=3, ncols=1, figsize=(11, 8), sharex=True, constrained_layout=True)

# use common y-limits for comparability
all_vals = np.concatenate([np.r_[v["seas"], v["90th thresh"], v["10th thresh"]] for v in curves_annual.values()])
ymin = float(np.nanmin(all_vals)) - 0.2
ymax = float(np.nanmax(all_vals)) + 0.2 

for ax, (name, ct) in zip(axes, curves_annual.items()):
    ax.plot(x_dates, ct["seas"],   label="Climatology (Annual)", color="C0", lw=1.6)
    ax.plot(x_dates, ct["90th thresh"], label="90th perc Threshold", ls="--", color= "#ff7f0e", lw=1.6)
    ax.plot(x_dates, ct["10th thresh"], label="10th perc Threshold", ls="--", color= "#24fa11", lw=1.6)
    ax.set_ylabel("Temp (°C)", fontweight="bold")
    ax.set_title(f"{name}: Long- Term Climatology, 90th and 10th perc Threshold ({CLIM_YEARS[0]}–{CLIM_YEARS[1]})", fontweight="bold")
    ax.legend(loc="upper left")
    ax.set_ylim(ymin, ymax)

year = 2000  
tick_dates = []
for m in range(1, 13):
    tick_dates.append(pd.Timestamp(year, m, 1))
    tick_dates.append(pd.Timestamp(year, m, 15))
tick_dates.append(pd.Timestamp(year, 12, 31))  

axes[-1].set_xticks(tick_dates)
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
axes[-1].set_xlim(x_dates[0], x_dates[-1])
axes[-1].set_xlabel("Day of Year", fontweight="bold")
fig.autofmt_xdate()


# Script for grid point analysis of MCS characterstics for Arabian Sea, Bey of Bengal and North Indian Ocean

In [ ]:
# Common file to compute the marine cold spells 
# import the packages and libraries
from glob import glob
from pathlib import Path
import numpy as np, pandas as pd, xarray as xr
import matplotlib.pyplot as plt
from dask.diagnostics import ProgressBar
import marineHeatWaves as mhw

# SST files path
FILES_GLOB = "/home/Desktop/Noah_data_1982-2024_SST_daily_mean/sst.day.mean.*.nc"
SST_VAR    = "sst"

# define the baseline year
BASELINE = (1982, 2024)

# Event definition based on Hobday et al. (2016)
MIN_DUR  = 5     # minimum event duration (days)
MAX_GAP  = 2     # join across gaps up to this many days

# Dask-friendly chunks
CHTIME, CHXY = 160, 60
 
# Set the regions
ROI_DICT = {
    "Arabian Sea": {"lon_min": 40.0, "lon_max": 80.0, "lat_min":  0.0, "lat_max": 30.0, "slug": "arabian_sea"},
    "Bay Of Bengal": {"lon_min": 80.0, "lon_max": 110.0,"lat_min":  0.0, "lat_max":  30.0, "slug": "bay_of_bengal"},
    "North Indian Ocean": {"lon_min": 40.0, "lon_max": 110.0,"lat_min":  0.0, "lat_max":  30.0,"slug": "north_indian_ocean"},
}

OUTROOT = Path("mcs_without_avg"); OUTROOT.mkdir(parents=True, exist_ok=True)

# open SST data using xarray
def open_sst(files_glob: str, roi: dict) -> tuple[xr.Dataset, str, str]:
    paths = sorted(glob(files_glob))
    if not paths:
        raise FileNotFoundError(f"No files match: {files_glob}")
    ds = xr.open_mfdataset(paths, combine="by_coords", chunks={"time": CHTIME}, engine="netcdf4")
    latn = "lat" if "lat" in ds.coords else "latitude"
    lonn = "lon" if "lon" in ds.coords else "longitude"
    ds = ds.sel({latn: slice(roi["lat_min"], roi["lat_max"]), lonn: slice(roi["lon_min"], roi["lon_max"])})
    return ds, latn, lonn


# use the hobdey's defination on region
def _detect_mask_1d(sst_1d: np.ndarray, thresh_1d: np.ndarray, min_dur: int = MIN_DUR, max_gap: int = MAX_GAP) -> np.ndarray:
    
    ok = np.isfinite(sst_1d) & np.isfinite(thresh_1d)
    exc = ok & (thresh_1d >= sst_1d)
    if not exc.any():
        return exc

    x = exc.astype(np.int8)
    n = x.size

    # join short gaps
    i = 0
    while i < n:
        if x[i] == 1:
            j = i + 1
            while j < n and x[j] == 1:
                j += 1
            g0 = j
            while g0 < n and x[g0] == 0:
                g0 += 1
            gap_len = g0 - j
            if gap_len > 0 and gap_len <= max_gap:
                x[j:g0] = 1
                j = g0
            i = j
        else:
            i += 1

    # enforce min duration
    y = x.copy()
    i = 0
    while i < n:
        if y[i] == 1:
            j = i + 1
            while j < n and y[j] == 1:
                j += 1
            if (j - i) < min_dur:
                y[i:j] = 0
            i = j
        else:
            i += 1

    return y.astype(bool)

def detect_mask_time(sst: xr.DataArray, thresh: xr.DataArray) -> xr.DataArray:
    
    return xr.apply_ufunc(_detect_mask_1d, sst, thresh, input_core_dims=[["time"], ["time"]], output_core_dims=[["time"]], vectorize=True, dask="parallelized", output_dtypes=[bool])

# per grid climatology & threshold 
def _clim_thresh_time_1d(ords_1d: np.ndarray, temp_1d: np.ndarray, y0: int, y1: int, pct: int, min_dur: int, max_gap: int) -> tuple[np.ndarray, np.ndarray]:
    
    # If mostly missing, return NaNs to avoid unstable fits
    if np.isfinite(temp_1d).sum() < 30:
        n = temp_1d.size
        return np.full(n, np.nan, float), np.full(n, np.nan, float)

    # MHW detection
    _, clim = mhw.detect(ords_1d.astype(int), temp_1d.astype(float),climatologyPeriod=[int(y0), int(y1)], pctile=int(pct), minDuration=int(min_dur),
                         joinAcrossGaps=True, maxGap=int(max_gap),)
    seas   = np.asarray(clim["seas"],   float)
    thresh = np.asarray(clim["thresh"], float)
    return seas, thresh

def build_grid_baseline(ds: xr.Dataset, latn: str, lonn: str,
                        pctile: int = 10) -> tuple[xr.DataArray, xr.DataArray]:
    
    t_index = pd.to_datetime(ds["time"].values)
    y0 = max(BASELINE[0], t_index.year.min())
    y1 = min(BASELINE[1], t_index.year.max())
    ords_da = xr.DataArray(np.array([d.toordinal() for d in t_index], dtype=int),coords={"time": ds["time"]},dims=["time"],).chunk({"time": -1})  
    sst = ds[SST_VAR].chunk({"time": -1, latn: CHXY, lonn: CHXY})

    seas_t, thresh_t = xr.apply_ufunc( _clim_thresh_time_1d, ords_da, sst,input_core_dims=[["time"], ["time"]], output_core_dims=[["time"], ["time"]], vectorize=True, dask="parallelized",
           output_dtypes=[float, float], kwargs=dict(y0=int(y0), y1=int(y1), pct=int(pctile), min_dur=MIN_DUR, max_gap=MAX_GAP) )
    seas_t   = seas_t.rename("seas_t")
    thresh_t = thresh_t.rename("thresh_t")
    return seas_t, thresh_t


**Arabian Sea MCS detection per grid**

In [ ]:
# This script detect and save the per-grid MCS and save in zarr files for Arabian Sea
REG = "Arabian Sea"
ROI = ROI_DICT[REG]

#  Open SST data
ds, latn, lonn = open_sst(FILES_GLOB, ROI)

#  Land mask & chunks
ocean = ds[SST_VAR].notnull().any("time")
sst   = ds[SST_VAR].where(ocean).chunk({"time": CHTIME, latn: CHXY, lonn: CHXY})

#  Per-grid climatology & 90th-threshold 
seas_t, thresh_t = build_grid_baseline(ds, latn, lonn, pctile=10)

# Detect Hobday events per grid 
# compute boolean mask for logic
evt_mask_bool = detect_mask_time(sst.chunk({"time": -1}),thresh_t.chunk({"time": -1})).rename("mcs_mask")  # bool

# NaN on land & keep float so NaNs persist to Zarr
evt_mask = evt_mask_bool.where(ocean).astype("float32")


# Daily metrics for future analysis
intensity = (seas_t - sst).rename("intensity")                
excess    = (thresh_t - sst).where(evt_mask == 1).rename("excess")  # only on event days

#  Yearly per-grid summaries
# compute starts on the boolean mask
starts_bool = (evt_mask_bool & ~(evt_mask_bool.shift(time=1, fill_value=False)))

# NaN on land, keep float dtype (so NaNs survive)
starts = starts_bool.where(ocean).astype("float32")

events_per_year_grid = (starts.groupby("time.year").sum("time").where(ocean).rename("events_per_year").astype("float32"))  
days_per_year_grid = (evt_mask.groupby("time.year").sum("time").where(ocean).rename("days_per_year").astype("float32"))
# mean_intensity_year_grid = (intensity.where(evt_mask == 1).groupby("time.year").mean("time").rename("mean_intensity_year").astype("float32"))
cumulative_intensity_year_grid = (intensity.where(evt_mask == 1).groupby("time.year").sum("time").rename("cumulative_intensity_year").astype("float32"))

# Save the zarr files
out_dir = OUTROOT / ROI_DICT[REG]["slug"]; out_dir.mkdir(parents=True, exist_ok=True)

daily_ds = xr.Dataset({"mcs_mask": evt_mask, "intensity": intensity, "excess": excess,"seas_t": seas_t, "thresh_t": thresh_t},coords={"time": ds["time"], latn: ds[latn], lonn: ds[lonn]},
                       ).chunk({"time": CHTIME, latn: CHXY, lonn: CHXY})

# Keep only core coords to avoid Zarr alignment issues
keep = {"time", latn, lonn}
dropc = [c for c in daily_ds.coords if c not in keep]
if dropc:
    daily_ds = daily_ds.reset_coords(dropc, drop=True)

print("Zarr file generation has started, wait for completion")
with ProgressBar():
    daily_ds.to_zarr(out_dir / "mcs_daily.zarr", mode="w", safe_chunks=False, align_chunks=True)
    events_per_year_grid.to_zarr(out_dir / "mcs_events_per_year_grid.zarr", mode="w", safe_chunks=False, align_chunks=True)
    days_per_year_grid.to_zarr(out_dir / "mcs_days_per_year_grid.zarr", mode="w", safe_chunks=False, align_chunks=True)
    # mean_intensity_year_grid.to_zarr(out_dir / "mcs_mean_intensity_year_grid.zarr", mode="w", safe_chunks=False, align_chunks=True)
    cumulative_intensity_year_grid.to_zarr(out_dir / "mcs_cumulative_intensity_year_grid.zarr", mode="w", safe_chunks=False, align_chunks=True)

print(f"{REG} MCS detection completed.")


**Arabian Sea MCS characterstics bar graphs**

In [ ]:
# the script plots the MCS event and days bar graphs using MCS zarr data of Arabian Sea
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats
import statsmodels.api as sm

# Defire the time period
BASELINE = (1982, 2024)
REG = "Arabian Sea"


# Load the zarr files
ds_event_path = xr.open_zarr("/home/Desktop/Jupyter files/arabian_sea/mcs_events_per_year_grid.zarr")
ds_days_path = xr.open_zarr("/home/Desktop/Jupyter files/arabian_sea/mcs_days_per_year_grid.zarr")

# Corrected coordinate extraction
latn = "lat" if "lat" in ds_event_path.coords else "latitude"
lonn = "lon" if "lon" in ds_event_path.coords else "longitude"

# Extract the specific variable array from the loaded Dataset
events_per_year = ds_event_path["events_per_year"]
days_per_year  =  ds_days_path["days_per_year"]

# Recreate the ocean mask (True for ocean, False for land)
ocean = events_per_year.notnull().any("year")

# X-axis year tick 
YEAR_TICK_STEP  = 3
YEAR_TICK_START = None
YEAR_TICK_END   = None
YEAR_TICK_ROT   = 0

# Area Weights Calculation
def area_weights_1d(ds_event_path: xr.Dataset, latn: str) -> xr.DataArray:
    return np.cos(np.deg2rad(ds_event_path[latn]))

def apply_year_ticks(ax, years, step=YEAR_TICK_STEP, start=YEAR_TICK_START, end=YEAR_TICK_END, rotate=YEAR_TICK_ROT):
    years = np.asarray(years, dtype=int)
    y_min, y_max = int(years.min()), int(years.max())
    a = y_min if start is None else int(start)
    b = y_max if end   is None else int(end)
    if (step is None) or (step == 0):
        ticks = np.unique(years)
    else:
        ticks = np.arange(a, b + 1, int(step), dtype=int)
    ax.set_xticks(ticks)
    ax.set_xticklabels([str(y) for y in ticks], rotation=rotate)

# calculte the Basin area avarage
# Calculate weights
w_lat = area_weights_1d(ds_event_path, latn)

# Area-weighted regional means (per year)
freq_region = (events_per_year.where(ocean)).weighted(w_lat).mean(dim=[latn, lonn]).compute()
days_region = (days_per_year.where(ocean)).weighted(w_lat).mean(dim=[latn, lonn]).compute()

total_events_region = float(freq_region.sum().values)
total_days_region   = float(days_region.sum().values)

print(f"{REG} totals — events: {total_events_region:.1f}, days: {total_days_region:.1f}")

# Plot the graphs
years_freq = (freq_region.coords.get("year", None).values
              if "year" in freq_region.coords
              else freq_region.get_index("year").values).astype(int)
years_days = (days_region.coords.get("year", None).values
              if "year" in days_region.coords
              else days_region.get_index("year").values).astype(int)


In [ ]:
# MCS Events and days per decade bar graphs with trends
def plot_trendlines(ax, x,y):
    y= np.asarray(y)
    
    # Theil-Sen non-parametric trend
    ts_res= stats.mstats.theilslopes(y,x, 0.95)
    slop_ts= ts_res[0]
    intercept_ts= ts_res[1]
    line_ts = (slop_ts*x) + intercept_ts
    # mann-kendall significance test 
    tau, p_ts = stats.kendalltau(x,y)
    # p value test
    p_ts_str = "<0.01" if p_ts < 0.01 else f"{p_ts:.3f}"
    label_ts =  f"Theil-Sen Trend: {slop_ts*10:.2f}/decade ($p={p_ts_str}$)"
    # plot the trend line 
    ax.plot(x, line_ts, color= "black", linestyle= '--', linewidth= 2, label= label_ts)
    
    # Poisson generalized linear model (GLM) trend 
    # Statsmodels requires explicitly add a constant for the intercept
    X_glm = sm.add_constant(x)
    glm_model = sm.GLM(y, X_glm, family=sm.families.Poisson())
    glm_results = glm_model.fit()
    # p value test
    p_glm = glm_results.pvalues[1]
    p_glm_str = "< 0.01" if p_glm < 0.01 else f"{p_glm:.3f}"
    # plot the trend line
    line_glm = glm_results.predict(X_glm)
    
    # The GLM slop will be in change per decade.
    slope_coef = glm_results.params[1]
    mean_y = np.mean(y)
    decadal_increase = slope_coef * mean_y * 10
    label_glm = f"Poisson GLM: +{decadal_increase:.2f}/decade ($p={p_glm_str}$)"
    ax.plot(x, line_glm, color= "green", linestyle= '--', linewidth= 2, label= label_glm)

    # add legends
    ax.legend()

# plot the Events bar chart with all trend
fig, ax1 = plt.subplots(figsize=(12, 4))
ax1.bar(years_freq, freq_region.values, color="#5b8def", width=0.8)
# add trend line
plot_trendlines(ax1,years_freq,freq_region)
ax1.set_title(f" MCS Events Per Year in {REG} ({BASELINE[0]}–{BASELINE[1]})", fontweight= "bold")
ax1.set_xlabel("Year", fontweight= "bold")
ax1.set_ylabel("MCS Events Per Year", fontweight= "bold")
ax1.grid(True, ls="--", alpha=0.4)
ax1.text(0.5, 0.92, f"Total Events: {total_events_region:.1f}", transform=ax1.transAxes, ha="center")
apply_year_ticks(ax1, years_freq)  
plt.tight_layout() 
plt.show()

# plot the days bar chart with all trend
fig, ax2 = plt.subplots(figsize=(12, 4))
ax2.bar(years_days, days_region.values, color="#f6a300", width=0.8)
# add trend line
plot_trendlines(ax2,years_days, days_region)
ax2.set_title(f"MCS Days Per Year in {REG} ({BASELINE[0]}–{BASELINE[1]})", fontweight= "bold")
ax2.set_xlabel("Year", fontweight= "bold") 
ax2.set_ylabel("MCS Days Per Year", fontweight= "bold")
ax2.grid(True, ls="--", alpha=0.4)
ax2.text(0.5, 0.92, f"Total MCS days: {total_days_region:.1f}", transform=ax2.transAxes, ha="center")
apply_year_ticks(ax2, years_days) 
plt.tight_layout()
plt.show()


**Two-Sample Kolmogorov-Smirnov (K-S) Test for distribution shift between two time periods**

In [ ]:
# Two-Sample Kolmogorov-Smirnov (K-S) Test for distribution shift between two time periods (1982–2002 vs 2003–2024)
import scipy.stats as stats
import numpy as np

#  Define the split year 
split_year = 2002

years_freq = (freq_region.coords.get("year", None).values
              if "year" in freq_region.coords
              else freq_region.get_index("year").values).astype(int)
years_days = (days_region.coords.get("year", None).values
              if "year" in days_region.coords
              else days_region.get_index("year").values).astype(int)


# Create boolean masks to split the data into two time periods (1982–2002 and 2003–2024)
mask_era1_event = (years_freq >= 1982) & (years_freq <= split_year)
mask_era2_event = (years_freq > split_year) & (years_freq <= 2024)

mask_era1_days = (years_days >= 1982) & (years_days <= split_year)
mask_era2_days = (years_days > split_year) & (years_days <= 2024)

# Extract the continuous data 
freq_region_era1 = freq_region.values[mask_era1_event]
freq_region_era2 = freq_region.values[mask_era2_event]
days_region_era1 = days_region.values[mask_era1_days]
days_region_era2 = days_region.values[mask_era2_days]

# Run the Two-Sample K-S Test

# for MHW Events
ks_statistic, p_value_ks = stats.ks_2samp(freq_region_era1, freq_region_era2)

print(f"Two-Sample Kolmogorov-Smirnov Test Results for MCS Events in {REG}:")
print(f"Period 1 (1982-{split_year}) vs Period 2 ({split_year+1}-2024)")
print(f"K-S Statistic (D): {ks_statistic:.4f}")
print(f"P-value: {p_value_ks:.4e}")

if p_value_ks < 0.05:
    print("Conclusion: The distribution of MCS Events has shifted significantly between the 1982-2002 and 2003-2024 periods.")
else:
    print("Conclusion: No statistically significant shift in the overall distribution.")

# for MHW Days
ks_statistic, p_value_ks = stats.ks_2samp(days_region_era1, days_region_era2)

print(f"\nTwo-Sample Kolmogorov-Smirnov Test Results for MCS Days in {REG}:")
print(f"Period 1 (1982-{split_year}) vs Period 2 ({split_year+1}-2024)")
print(f"K-S Statistic (D): {ks_statistic:.4f}")
print(f"P-value: {p_value_ks:.4e}")

if p_value_ks < 0.05:
    print("Conclusion: The distribution of MCS Days has shifted significantly between 1982-2002 and 2003-2024 periods.")
else:
    print("Conclusion: No statistically significant shift in the overall distribution.")

In [ ]:
# the script plots the MCS cumulative intensity bar graphs using MCS zarr data of Arabian Sea
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats
import statsmodels.api as sm

# Defire the time period
BASELINE = (1982, 2024)
REG = "Arabian Sea"


# Load the zarr files
cum_int_as = xr.open_zarr("/home/Desktop/Jupyter files/arabian_sea/mcs_cumulative_intensity.zarr")

# Corrected coordinate extraction
latn = "lat" if "lat" in cum_int_as.coords else "latitude"
lonn = "lon" if "lon" in cum_int_as.coords else "longitude"


# Recreate the ocean mask (True for ocean, False for land)
ocean = cum_int_as.notnull().any("year")

# X-axis year tick 
YEAR_TICK_STEP  = 3
YEAR_TICK_START = None
YEAR_TICK_END   = None
YEAR_TICK_ROT   = 0

# Area Weights Calculation
def area_weights_1d(cum_int_as: xr.Dataset, latn: str) -> xr.DataArray:
    return np.cos(np.deg2rad(cum_int_as[latn]))

def apply_year_ticks(ax, years, step=YEAR_TICK_STEP, start=YEAR_TICK_START, end=YEAR_TICK_END, rotate=YEAR_TICK_ROT):
    years = np.asarray(years, dtype=int)
    y_min, y_max = int(years.min()), int(years.max())
    a = y_min if start is None else int(start)
    b = y_max if end   is None else int(end)
    if (step is None) or (step == 0):
        ticks = np.unique(years)
    else:
        ticks = np.arange(a, b + 1, int(step), dtype=int)
    ax.set_xticks(ticks)
    ax.set_xticklabels([str(y) for y in ticks], rotation=rotate)

# calculte the Basin area avarage
# Calculate weights
w_lat = area_weights_1d(cum_int_as, latn)

# Area-weighted regional means (per year)
cum_int_region_as = (cum_int_as["events_per_year"].where(ocean)).weighted(w_lat).mean(dim=[latn, lonn]).compute()

# Plot the graphs
years_cum_int = (cum_int_region_as.coords.get("year", None).values
              if "year" in cum_int_region_as.coords
              else cum_int_region_as.get_index("year").values).astype(int)


In [ ]:
# MCS Events and days per decade bar graphs with trends
def plot_trendlines(ax, x,y):
    y= np.asarray(y)

    # Theil-Sen non-parametric trend
    ts_res= stats.mstats.theilslopes(y,x, 0.95)
    slop_ts= ts_res[0]
    intercept_ts= ts_res[1]
    line_ts = (slop_ts*x) + intercept_ts
    # mann-kendall significance test 
    tau, p_ts = stats.kendalltau(x,y)
    # p value test
    p_ts_str = "<0.01" if p_ts < 0.01 else f"{p_ts:.3f}"
    label_ts =  f"Theil-Sen Trend: {slop_ts*10:.2f}/decade ($p={p_ts_str}$)"
    # plot the trend line 
    ax.plot(x, line_ts, color= "black", linestyle= '--', linewidth= 2, label= label_ts)
    
    # Poisson generalized linear model (GLM) trend 
    # Statsmodels requires explicitly add a constant for the intercept
    X_glm = sm.add_constant(x)
    glm_model = sm.GLM(y, X_glm, family=sm.families.Poisson())
    glm_results = glm_model.fit()
    # p value test
    p_glm = glm_results.pvalues[1]
    p_glm_str = "< 0.01" if p_glm < 0.01 else f"{p_glm:.3f}"
    # plot the trend line
    line_glm = glm_results.predict(X_glm)
    
    # The GLM slop will be in change per decade.
    slope_coef = glm_results.params[1]
    mean_y = np.mean(y)
    decadal_increase = slope_coef * mean_y * 10
    label_glm = f"Poisson GLM: +{decadal_increase:.2f}/decade ($p={p_glm_str}$)"
    ax.plot(x, line_glm, color= "green", linestyle= '--', linewidth= 2, label= label_glm)

    # add legends
    ax.legend()

# plot the Events bar chart with all trend
fig, ax1 = plt.subplots(figsize=(12, 4))
ax1.bar(years_cum_int, cum_int_region_as.values, color="#5b8def", width=0.8)
# add trend line
plot_trendlines(ax1,years_cum_int,cum_int_region_as)
ax1.set_title(f" MCS Cumulative Intensity Per Year in {REG} ({BASELINE[0]}–{BASELINE[1]})", fontweight= "bold")
ax1.set_xlabel("Year", fontweight= "bold")
ax1.set_ylabel("MCS Cumulative Intensity Per Year", fontweight= "bold")
ax1.grid(True, ls="--", alpha=0.4)
apply_year_ticks(ax1, years_cum_int)  
plt.tight_layout() 
plt.show()


**Bay Of Bengal MCS characterstics bar graphs**

In [ ]:
# the script plots the MCS event and days bar graphs using MCS zarr data of Bay Of Bengal
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats
import statsmodels.api as sm

# Defire the time period
BASELINE = (1982, 2024)
REG = "Bay Of Bengal"


# Load the zarr files
ds_event_path = xr.open_zarr("/home/Desktop/Jupyter files/bay_of_bengal/mcs_events_per_year_grid.zarr")
ds_days_path = xr.open_zarr("/home/Desktop/Jupyter files/bay_of_bengal/mcs_days_per_year_grid.zarr")

# Corrected coordinate extraction
latn = "lat" if "lat" in ds_event_path.coords else "latitude"
lonn = "lon" if "lon" in ds_event_path.coords else "longitude"

# Extract the specific variable array from the loaded Dataset
events_per_year = ds_event_path["events_per_year"]
days_per_year  =  ds_days_path["days_per_year"]

# Recreate the ocean mask (True for ocean, False for land)
ocean = events_per_year.notnull().any("year")

# X-axis year tick 
YEAR_TICK_STEP  = 3
YEAR_TICK_START = None
YEAR_TICK_END   = None
YEAR_TICK_ROT   = 0

# Area Weights Calculation
def area_weights_1d(ds_event_path: xr.Dataset, latn: str) -> xr.DataArray:
    return np.cos(np.deg2rad(ds_event_path[latn]))

def apply_year_ticks(ax, years, step=YEAR_TICK_STEP, start=YEAR_TICK_START, end=YEAR_TICK_END, rotate=YEAR_TICK_ROT):
    years = np.asarray(years, dtype=int)
    y_min, y_max = int(years.min()), int(years.max())
    a = y_min if start is None else int(start)
    b = y_max if end   is None else int(end)
    if (step is None) or (step == 0):
        ticks = np.unique(years)
    else:
        ticks = np.arange(a, b + 1, int(step), dtype=int)
    ax.set_xticks(ticks)
    ax.set_xticklabels([str(y) for y in ticks], rotation=rotate)

# calculte the Basin area avarage
# Calculate weights
w_lat = area_weights_1d(ds_event_path, latn)

# Area-weighted regional means (per year)
freq_region = (events_per_year.where(ocean)).weighted(w_lat).mean(dim=[latn, lonn]).compute()
days_region = (days_per_year.where(ocean)).weighted(w_lat).mean(dim=[latn, lonn]).compute()

total_events_region = float(freq_region.sum().values)
total_days_region   = float(days_region.sum().values)

print(f"{REG} totals — events: {total_events_region:.1f}, days: {total_days_region:.1f}")

# Plot the graphs
years_freq = (freq_region.coords.get("year", None).values
              if "year" in freq_region.coords
              else freq_region.get_index("year").values).astype(int)
years_days = (days_region.coords.get("year", None).values
              if "year" in days_region.coords
              else days_region.get_index("year").values).astype(int)


In [ ]:
# MCS Events and days per decade bar graphs with trends
def plot_trendlines(ax, x,y):
    y= np.asarray(y)
    
    # Theil-Sen non-parametric trend
    ts_res= stats.mstats.theilslopes(y,x, 0.95)
    slop_ts= ts_res[0]
    intercept_ts= ts_res[1]
    line_ts = (slop_ts*x) + intercept_ts
    # mann-kendall significance test 
    tau, p_ts = stats.kendalltau(x,y)
    # p value test
    p_ts_str = "<0.01" if p_ts < 0.01 else f"{p_ts:.3f}"
    label_ts =  f"Theil-Sen Trend: {slop_ts*10:.2f}/decade ($p={p_ts_str}$)"
    # plot the trend line 
    ax.plot(x, line_ts, color= "black", linestyle= '--', linewidth= 2, label= label_ts)
    
    # Poisson generalized linear model (GLM) trend 
    # Statsmodels requires explicitly add a constant for the intercept
    X_glm = sm.add_constant(x)
    glm_model = sm.GLM(y, X_glm, family=sm.families.Poisson())
    glm_results = glm_model.fit()
    # p value test
    p_glm = glm_results.pvalues[1]
    p_glm_str = "< 0.01" if p_glm < 0.01 else f"{p_glm:.3f}"
    # plot the trend line
    line_glm = glm_results.predict(X_glm)
    
    # The GLM slop will be in change per decade.
    slope_coef = glm_results.params[1]
    mean_y = np.mean(y)
    decadal_increase = slope_coef * mean_y * 10
    label_glm = f"Poisson GLM: +{decadal_increase:.2f}/decade ($p={p_glm_str}$)"
    ax.plot(x, line_glm, color= "green", linestyle= '--', linewidth= 2, label= label_glm)

    # add legends
    ax.legend()

# plot the Events bar chart with all trend
fig, ax1 = plt.subplots(figsize=(12, 4))
ax1.bar(years_freq, freq_region.values, color="#5b8def", width=0.8)
# add trend line
plot_trendlines(ax1,years_freq,freq_region)
ax1.set_title(f" MCS Events Per Year in {REG} ({BASELINE[0]}–{BASELINE[1]})", fontweight= "bold")
ax1.set_xlabel("Year", fontweight= "bold")
ax1.set_ylabel("MCS Events Per Year", fontweight= "bold")
ax1.grid(True, ls="--", alpha=0.4)
ax1.text(0.5, 0.92, f"Total Events: {total_events_region:.1f}", transform=ax1.transAxes, ha="center")
apply_year_ticks(ax1, years_freq)  
plt.tight_layout() 
plt.show()

# plot the days bar chart with all trend
fig, ax2 = plt.subplots(figsize=(12, 4))
ax2.bar(years_days, days_region.values, color="#f6a300", width=0.8)
# add trend line
plot_trendlines(ax2,years_days, days_region)
ax2.set_title(f"MCS Days Per Year in {REG} ({BASELINE[0]}–{BASELINE[1]})", fontweight= "bold")
ax2.set_xlabel("Year", fontweight= "bold") 
ax2.set_ylabel("MCS Days Per Year", fontweight= "bold")
ax2.grid(True, ls="--", alpha=0.4)
ax2.text(0.5, 0.92, f"Total MCS days: {total_days_region:.1f}", transform=ax2.transAxes, ha="center")
apply_year_ticks(ax2, years_days) 
plt.tight_layout()
plt.show()


In [ ]:
# The script plots the MCS Cumulative intensity bar with trend lined per decades and per year using MCS zarr data of Bay Of Bengal
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats
import statsmodels.api as sm

# Defire the time period
BASELINE = (1982, 2024)
REG = "Bay Of Bengal"

# Load the zarr files
ds_cum_int= xr.open_zarr("/home/Desktop/Jupyter files/bay_of_bengal/mcs_cumulative_intensity_year_grid.zarr")

# Corrected coordinate extraction
latn = "lat" if "lat" in ds_cum_int.coords else "latitude"
lonn = "lon" if "lon" in ds_cum_int.coords else "longitude"

# Extract the specific variable array from the loaded Dataset
cumulative_intensity_year= ds_cum_int["cumulative_intensity_year"]

# Recreate the ocean mask (True for ocean, False for land)
ocean = cumulative_intensity_year.notnull().any("year")

# X-axis year tick
YEAR_TICK_STEP  = 3
YEAR_TICK_START = None
YEAR_TICK_END   = None
YEAR_TICK_ROT   = 0

# Area Weights Calculation
def area_weights_1d(ds_cum_int: xr.Dataset, latn: str) -> xr.DataArray:
    return np.cos(np.deg2rad(ds_cum_int[latn]))

def apply_year_ticks(ax, years, step=YEAR_TICK_STEP, start=YEAR_TICK_START, end=YEAR_TICK_END, rotate=YEAR_TICK_ROT):
    years = np.asarray(years, dtype=int)
    y_min, y_max = int(years.min()), int(years.max())
    a = y_min if start is None else int(start)
    b = y_max if end   is None else int(end)
    if (step is None) or (step == 0):
        ticks = np.unique(years)
    else:
        ticks = np.arange(a, b + 1, int(step), dtype=int)
    ax.set_xticks(ticks)
    ax.set_xticklabels([str(y) for y in ticks], rotation=rotate)

# calculte the Basin area avarage
# Calculate weights
w_lat = area_weights_1d(ds_cum_int, latn)

# Mean, max and cumulative intensity per year
cum_int_region = cumulative_intensity_year.where(ocean).weighted(w_lat).mean(dim=[latn, lonn], skipna= True).compute()

In [ ]:
# Per decade MCS intensity bar graphs with trends
def plot_trendlines(ax, x,y):
    y= np.asarray(y)
    
    # Theil-Sen non-parametric trend
    ts_res= stats.mstats.theilslopes(y,x, 0.95)
    slop_ts= ts_res[0]
    intercept_ts= ts_res[1]
    line_ts = (slop_ts*x) + intercept_ts
    # mann-kendall significance test 
    tau, p_ts = stats.kendalltau(x,y)
    # p value test
    p_ts_str = "<0.01" if p_ts < 0.01 else f"{p_ts:.3f}"
    label_ts =  f"Theil-Sen Trend: {slop_ts*10:.2f}/decade ($p={p_ts_str}$)"
    # plot the trend line 
    ax.plot(x, line_ts, color= "black", linestyle= '--', linewidth= 2, label= label_ts)
    
    # Poisson generalized linear model (GLM) trend 
    # Statsmodels requires explicitly add a constant for the intercept
    X_glm = sm.add_constant(x)
    glm_model = sm.GLM(y, X_glm, family=sm.families.Poisson())
    glm_results = glm_model.fit()
    # p value test
    p_glm = glm_results.pvalues[1]
    p_glm_str = "< 0.01" if p_glm < 0.01 else f"{p_glm:.3f}"
    # plot the trend line
    line_glm = glm_results.predict(X_glm)
    
    # The GLM slop will be in change per decade.
    slope_coef = glm_results.params[1]
    mean_y = np.mean(y)
    decadal_increase = slope_coef * mean_y * 10
    label_glm = f"Poisson GLM: {decadal_increase:.2f}/decade ($p={p_glm_str}$)"
    ax.plot(x, line_glm, color= "red", linestyle= '--', linewidth= 2, label= label_glm)

    # add legends
    ax.legend()


# Plotting the bar graphs

years_cum_int = (cum_int_region.coords.get("year", None).values
              if "year" in cum_int_region.coords
              else cum_int_region.get_index("year").values).astype(int)

# Cumulative intensity
fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(years_cum_int, cum_int_region, color= "#009E73", width=0.8)
# add trend line
plot_trendlines(ax, years_cum_int, cum_int_region)
ax.set_title(f"Cumulative MCS Intensity in {REG} ({BASELINE[0]}–{BASELINE[1]})", fontweight= "bold")
ax.set_xlabel("Year", fontweight= "bold")
ax.set_ylabel("Intensity (°C·days)", fontweight= "bold")
apply_year_ticks(ax, years_cum_int)
ax.grid(True, ls="--", alpha=0.4)
plt.tight_layout()
plt.show()
